In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx

import os
import logging
from datetime import datetime

In [2]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [5]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [49]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [50]:
print(qat_model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

#### QAT via torch.ao.quantization.prepare_qat (Eager way, older)

In [55]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [52]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [53]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [54]:
qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)
print(qat_model_prepared)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(
    64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): QuantizableBasicBlock(
      (conv1): Conv2d(
        64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
          fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint

/tmp/ipykernel_79441/224494645.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)


#### QAT via prepare_qat_fx (New (FX) way. lets you fine-tune per-module quantization in a declarative way)

In [57]:
example_input = torch.randn(1, 3, 32, 32) # For checking 

qconfig_dict = {
    "": default_qconfig,  # default for all layers
    "module_name": [("conv1", None), ("fc", None)]  # disable quant for first and last
}

qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
print(qat_model_prepared)

/tmp/ipykernel_79441/4015603833.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
/home/bohdan/RAI/rai-env/lib/python3.12/site-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. P

GraphModule(
  (conv1): ConvBnReLU2d(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (activation_post_process_0): FusedMovingAvgObsFakeQuantize(
    fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
    (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
  )
  (maxpool): Identity()
  (activation_post_process_1): FusedMovingAvgObsFakeQuantize(
    fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
    (activation_post_process): MovingAverageMinMaxObserver(mi

#### QAT Model traininig

In [58]:
qat_model_prepared.to(device)
start_of_training_timestamp = training_loop(qat_model_prepared, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=5)

2025-10-07 13:46:31,585 [INFO] Epoch [1/5] Train Loss: 1.3427, Train Acc: 50.97% Test Loss: 1.2349, Test Acc: 56.39%
2025-10-07 13:47:18,633 [INFO] Epoch [2/5] Train Loss: 0.8803, Train Acc: 68.98% Test Loss: 0.8666, Test Acc: 70.17%
2025-10-07 13:48:09,138 [INFO] Epoch [3/5] Train Loss: 0.6898, Train Acc: 75.80% Test Loss: 0.7279, Test Acc: 75.53%
2025-10-07 13:48:56,917 [INFO] Epoch [4/5] Train Loss: 0.5801, Train Acc: 79.75% Test Loss: 0.6582, Test Acc: 78.11%
2025-10-07 13:49:45,620 [INFO] Epoch [5/5] Train Loss: 0.5038, Train Acc: 82.57% Test Loss: 0.5597, Test Acc: 81.41%


In [59]:
qat_model_prepared.eval()
qat_model_prepared.to("cpu")
final_quantized_model = convert_fx(qat_model_prepared)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(final_quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

/tmp/ipykernel_79441/1044601629.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  final_quantized_model = convert_fx(qat_model_prepared)


Quantized model saved as ./trained_models/resnet18_cifar10_qat_07.10.2025-13:45:40.pth
Quantized file size (MB): 10.803154945373535


In [60]:
print(final_quantized_model)

GraphModule(
  (conv1): ConvBnReLU2d(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): Identity()
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.06429646164178848, zero_point=0, padding=(1, 1))
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.14538443088531494, zero_point=68, padding=(1, 1))
    )
    (1): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.06418824940919876, zero_point=0, padding=(1, 1))
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.1206456869840622, zero_point=67, padding=(1, 1))
    )
  )
  (layer2): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 128, kernel_size=(3, 3), stride=(2, 2), scale=0.0596

In [68]:
qat_model.eval()
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = qat_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
logging.info(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)

RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor